![](https://github.com/destination-earth/DestinE-DataLake-Lab/blob/main/img/DestinE-banner.jpg?raw=true)

# Extraction of ClimateDT data
Data portfolio: https://confluence.ecmwf.int/display/DDCZ/Climate+DT+Phase+1+data+catalogue#ClimateDTPhase1datacatalogue-Fieldsonasinglelevelorsurface

### Connect to polytope

Enter your DESP credentials when prompted by the authentication cell below.

In [1]:
%%capture cap
%run ./src/desp-authentication.py

In [2]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]
access_token

'Token successfully written to /home/koenifra/.polytopeapirc'

In [3]:
import pandas as pd

### Load csv

In [4]:
geosphere_stations = pd.read_csv("./location_hourly_daily_stations_2007_2025.csv")
geosphere_stations

,id,Stationsname,Lon,Lat,Höhe [m],Startdatum,Enddatum,Bundesland
0,1,Aflenz,15.240690,47.545940,783.2,1983-05-01 00:00:00+00:00,2100-12-31 00:00:00+00:00,Steiermark
1,2,Aigen im Ennstal,14.138260,47.532780,641.0,1939-03-01 00:00:00+00:00,2100-12-31 00:00:00+00:00,Steiermark
2,3,Allentsteig,15.366940,48.690830,598.8,1983-10-01 00:00:00+00:00,2100-12-31 00:00:00+00:00,Niederösterreich
3,4,Amstetten,14.895000,48.108890,266.0,1936-01-01 00:00:00+00:00,2100-12-31 00:00:00+00:00,Niederösterreich
4,5,Bad Aussee,13.758440,47.610500,743.1,1983-09-01 00:00:00+00:00,2100-12-31 00:00:00+00:00,Steiermark
...,...,...,...,...,...,...,...,...
624,12500,Bischofshofen,13.216670,47.400000,550.0,1980-06-02 13:00:00+00:00,2008-06-30 23:00:00+00:00,Salzburg
625,14520,Prutz,10.666667,47.066666,870.0,1967-07-01 00:00:00+00:00,2008-01-31 23:00:00+00:00,Tirol
626,15600,Obertauern,13.566670,47.266670,1742.0,1984-08-23 14:00:00+00:00,2008-06-30 23:00:00+00:00,Salzburg
627,19600,St. Lorenzen im Lesachtal,12.783334,46.700001,1260.0,1980-01-01 00:00:00+00:00,2007-12-31 23:00:00+00:00,Kärnten


In [5]:
scenarios = pd.read_csv("./climate-dt-scenarios.csv", sep=";")
scenarios

,params,model,level type,experiment,resolution,temporal extent,activity,typeOfSimulation
0,167/260048,ICON,sfc,hist,high,1990-2024,CMIP6,NaN
1,141/167/228/260048,IFS-NEMO,sfc,hist,high,1990-2024,CMIP6,NaN
2,228141,IFS-NEMO,sol,hist,high,1990-2024,CMIP6,NaN
3,167/228,IFS-FESOM,sfc,cont,high,1990-2004,HighResMIP,Control simulation
4,167/228,IFS-NEMO,sfc,cont,high,1990-2007,HighResMIP,Control simulation
5,167/260048,ICON,sfc,SSP3-7.0,high,2020-2039,ScenarioMIP,Future projection
6,167/228,IFS-FESOM,sfc,SSP3-7.0,high,2020-2039,ScenarioMIP,Future projection
7,141/167/228/260048,IFS-NEMO,sfc,SSP3-7.0,high,2020-2039,ScenarioMIP,Future projection
8,228141,IFS-NEMO,sol,SSP3-7.0,high,2020-2050,ScenarioMIP,NaN
9,167/228,IFS-FESOM,sfc,cont,high,2017-2023,story-nudging,Storyline simulation


**The following dependencies are missing in the current kernel**

In [6]:
# pip install -q polytope-client covjsonkit

In [7]:
from typing import Literal
def extract_cdt_ts(experiment: str,
                   activity: str,
                   level_type: str,
                   datestring: str,
                   model: str,
                   parameter: str,
                   location: list,
                   feature: Literal["timeseries", "polygon"]="timeseries",
                   time_resolution: str="0000/to/2300",
                   resolution: str="high"):
    import earthkit.data
    
    if feature == "timeseries":
        feature_dict = {
            "type" : "timeseries",
            "points": location,
            "time_axis": "date"
        }
    elif feature == "polygon":
        feature_dict = {
            "type" : "polygon",
            "shape": location
        }
    else:
        raise TypeError("feature not supported")
    
    request = {
        # static parameters of climate dt data
        "class": "d1",
        "dataset": "climate-dt",
        "generation": "1",
        "expver": "0001",
        "stream": "clte",
        "type": "fc",
        # generic
        "activity": activity,
        "experiment": experiment,
        "levtype": level_type,
        "date": datestring,
        "model": model,
        "param": parameter,
        "param": "167",
        "realization": "1",
        "resolution": resolution,
        "time": time_resolution,
        "feature": feature_dict
    }

    # commented out to check if only one level is request it gets faster or not
    if level_type == "sol":
        # request["levelist"] = "1/to/5"
        request["levelist"] = "1"
    
    try:
        ds = earthkit.data.from_source("polytope", 
                                       "destination-earth", 
                                       request, stream=False, 
                                       address='polytope.lumi.apps.dte.destination-earth.eu')
    except Exception as e:
        print(e)
        
    return ds.to_xarray()
    
    

In [8]:
def get_datestring(temp_extent: str):
    '''
    Example output string "20200101/to/20210101"
    '''
    years = temp_extent.split("-")
    return f"{years[0]}0101/to/{years[1]}1231"
    

## Extract TS per Geosphere Station and scenario

In [9]:
def get_station_data(geosphere_station, scenario):
    # convert georeference
    latlon = [[float(geosphere_station["Lat"]),float(geosphere_station["Lon"])]]

    # get time series of station
    ts = extract_cdt_ts(scenario["experiment"],
                        scenario["activity"],
                        scenario["level type"],
                        get_datestring(scenario["temporal extent"]),
                        scenario["model"],
                        scenario["params"],
                        latlon)
    return ts.assign_coords({"stationid": geosphere_station["id"]}) \
        .expand_dims(dim="stationid")

In [11]:
import concurrent.futures
import xarray as xr
import os


scenario = scenarios.iloc[6]

scenario_ts = xr.Dataset()

In [12]:
scenario

params                        167/228
model                       IFS-FESOM
level type                        sfc
experiment                   SSP3-7.0
resolution                       high
temporal extent             2020-2039
activity                  ScenarioMIP
typeOfSimulation    Future projection
Name: 6, dtype: object

In [ ]:
store_path = f"{scenario['model']}_{scenario['level type']}_{scenario['activity']}_{scenario['experiment']}.zarr"
# store_path = f"{scenario['model']}_{scenario['level type']}_{scenario['activity']}_SSP3-7.zarr"
print(f"Data will be stored in: {store_path}")

Data will be stored in: ICON_sfc_CMIP6_hist.zarr


In [1]:
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    futures = []
    for ind, station in geosphere_stations.iloc[8:].iterrows():
    #for station in station_list:
        futures.append(executor.submit(get_station_data, geosphere_station=station, scenario=scenario))
    
    for future in concurrent.futures.as_completed(futures):
        if not os.path.exists(store_path):
            #scenario_ts = future.result().copy()
            future.result().chunk(chunks={"stationid": 1,
                              "latitude": 1,
                              "longitude": 1,
                              "levelist": 1,
                              "number": 1,
                              "datetime": 1,
                              "t": "auto"
                             }).to_zarr(store=store_path,
                        mode="w")
        else:
            #scenario_ts = xr.concat([scenario_ts, future.result()], dim="stationid")
            future.result().chunk(chunks={"stationid": 1,
                              "latitude": 1,
                              "longitude": 1,
                              "levelist": 1,
                              "number": 1,
                              "datetime": 1,
                              "t": "auto"
                             }).to_zarr(store=store_path,
                        append_dim="stationid")

NameError: name 'concurrent' is not defined

### Write zarr store which gets finally populated with data

In [ ]:
import concurrent.futures
import xarray as xr
import os


scenario = scenarios.iloc[4]

scenario_ts = xr.Dataset()

store_path = f"./{scenario['model']}_{scenario['level type']}_{scenario['activity']}.zarr"
print(f"Data will be stored in: {store_path}")

with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    futures = []
    for ind, station in geosphere_stations.iterrows():
    #for station in station_list:
        futures.append(executor.submit(get_station_data, geosphere_station=station, scenario=scenario))
    
    for future in concurrent.futures.as_completed(futures):
        if not os.path.exists(store_path):
            #scenario_ts = future.result().copy()
            future.result().chunk(chunks={"stationid": 1,
                              "latitude": 1,
                              "longitude": 1,
                              "levelist": 1,
                              "number": 1,
                              "datetime": 1,
                              "t": "auto"
                             }).to_zarr(store=store_path,
                        mode="w")
        else:
            #scenario_ts = xr.concat([scenario_ts, future.result()], dim="stationid")
            future.result().chunk(chunks={"stationid": 1,
                              "latitude": 1,
                              "longitude": 1,
                              "levelist": 1,
                              "number": 1,
                              "datetime": 1,
                              "t": "auto"
                             }).to_zarr(store=store_path,
                        append_dim="stationid")